# Cross-model tournament — 260723

Runs the shortlisted `260723` models against each other in inference and writes one behavioural-log JSON per matchup (the format `octagon_analysis` consumes).

**Where the code lives.** This notebook is config + visualisation only. Everything that runs or checks a matchup is in the two shared modules:
- `tournament/tournament_func.py` — assembling a matchup run dir, `run_matchup`, `run_tournament`.
- `tournament/tournament_checks.py` — `preflight`, `smoke_test`, `check_batch`, `rerun_matchups`.

Analysis of the resulting logs is `octagon_analysis/analysis/tournament_analysis.py` + `demos/tournament/tournament_analysis_260723.ipynb`.

**What is different about this batch.** The `260723` models were trained on `build_260723_01_social`, with the no-movement penalty **break removed** — every step is penalised, resting included. The tournament build (`build_260724_01_tournament`) has the break absent too, so for this batch the tournament reward structure **matches** what the models were trained under. That was not true of `260715` (trained with the break present), where the reward analysis had to be held back. So the reward ranking is worth reading here, not just the score ranking.

**Entrants are pinned at 3e6 steps.** `MODELS_ROOT` points at `results/2607/260723_frozen_3e6`, a snapshot of the eight entrants' checkpoints, so extending the live `260723` runs to 4e6 cannot change the policies mid-tournament. A 4e6 tournament would be a separate batch against a separate snapshot.

**Before running:**
1. Activate the conda env that has `mlagents` (the kernel for this notebook must be that env).
2. Sanity-check CUDA (`nvidia-smi`, `torch.cuda.is_available()`), following the driver notes in `CLAUDE.md`.
3. The build below must be the tournament build with `ScriptedTrialSequence.cs` + the timing keys; an older build silently ignores `--trial_seq` and generates random trials.

Actions are **sampled** (not greedy), matching training behaviour → matchups are not bit-reproducible; run enough episodes to average over sampling noise. Trial content/timing is pinned by the predetermined `trial_seq`.

In [8]:
import importlib
import sys
from pathlib import Path

# Repo root on sys.path so tournament.* and trainer_and_simulator_functions both resolve.
REPO_ROOT = Path("~/repos/agent_training").expanduser()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Reload all three modules. Python caches imports, so editing tournament_func.py does not
# affect a running kernel. The 260730 batch was lost this way: the fixed file was on disk but
# the kernel kept the old functions. If in any doubt, restart the kernel.
import trainer_and_simulator_functions as _tsf
importlib.reload(_tsf)
import tournament.tournament_func as tf
importlib.reload(tf)
import tournament.tournament_checks as tc
importlib.reload(tc)

from tournament.tournament_func import run_tournament

## Config

Everything batch-specific lives in this cell. `MODELS_ROOT` is the parent folder; entrants are its immediate subfolders (each with `OctagonAgentSocial/checkpoint.pt`). All entrants must live under one `MODELS_ROOT` — to mix `260715` and `260723`, gather the chosen run dirs under a single folder first.

In [9]:
OCTAGON_DIR = Path("~/Unity/Octagon").expanduser()

# Tournament build (headless Linux). The penalty break is absent, matching how the 260723
# models were trained, so tournament reward is comparable to training reward here.
UNITY_ENV_PATH = Path("~/Unity/Octagon/builds/260909/build_260909_01_tournament_nosteppenalty.x86_64").expanduser()

# Parent folder holding the entrant run dirs (each an immediate subfolder).
# This is the frozen snapshot, not the live run dirs. run_matchup re-reads each entrant's
# checkpoint.pt per matchup, so extending the live 260723 runs to 4e6 would mix 3e6 and 4e6
# policies within one tournament, and check_batch could not detect it.
MODELS_ROOT = Path("~/Unity/Octagon/results/2607/260723_frozen_3e6").expanduser()

# Hand-picked entrants, from the Stage-4 shortlist in
# octagon_analysis/demos/strategy_characterisation/competitive/model_characterisation_demo260723.ipynb.
# Listed central -> intermediate -> peripheral so the analysis matrices read in strategy order.
# Set to None to use every model under MODELS_ROOT (N models gives C(N,2)+N matchups).
RUN_IDS = [
    "260723_0034",  # central
    "260723_0039",  # central
    "260723_0044",  # intermediate (the only one that earned a place in the gap)
    "260723_0035",  # peripheral, top of the shortlist
    "260723_0001",  # peripheral
    "260723_0033",  # peripheral
    "260723_0051",  # peripheral
    "260723_0023",  # peripheral
]

# Predetermined trial sequence (pins trial content + timing across all matchups).
# Keep it longer than EPISODES, since the build stops when the sequence is exhausted.
TRIAL_SEQ = Path("~/repos/agent_training/trial_sequences/trials_100000_seed17.json").expanduser()

# Where the per-matchup JSON logs land: one folder per batch, so a round-robin is self-contained.
OUT_ROOT = Path("~/Unity/Octagon/simulations/tournaments/260723").expanduser()

# MLAgents episodes per matchup. Conversion (measured on 260715 inference): trials ~= episodes - 1.
# So for a target of T trials/matchup, set EPISODES ~= T + 1. Same value as 260715, so the two
# batches carry equal weight per matchup.
EPISODES = 350   # ~350 trials/matchup

# Run each distinct pair in both seat orders. P1/P2 is symmetric, so the function defaults
# this off. On here to double the data per pair (2 x ~350 = ~700 trials) and to re-check the
# seat has no effect. On 260715 the mean mirror gap was 0.018, so the assumption held.
INCLUDE_MIRRORS = True

# Include models playing against themselves, one per entrant, as a ~50% baseline check.
INCLUDE_SELF_MATCHUPS = False

In [10]:
# Fail fast on a missing path, a missing checkpoint, or a generated config with no
# environment_parameters/step_penalty. Without that block the build throws on every action
# and the matchup becomes a walkover that still writes a healthy-looking JSON. The check runs
# from here rather than inside tournament_func, so it also catches a stale kernel.
n_matchups = tc.preflight(
    models_root=MODELS_ROOT,
    unity_env_path=UNITY_ENV_PATH,
    trial_seq=TRIAL_SEQ,
    run_ids=RUN_IDS,                        # None => every model under MODELS_ROOT
    include_mirrors=INCLUDE_MIRRORS,
    include_self_matchups=INCLUDE_SELF_MATCHUPS,
    base_port=9100,                     # match the round-robin call below, so preflight
                                        # checks the whole span is free
)

All paths OK, config carries step_penalty. 8 entrants => 56 matchups (with mirrors, no self-matchups).


## (Optional) Single-matchup smoke test

Run one matchup with a few episodes first, to confirm the build + checkpoints load and both agents actually play, before committing to the full round-robin (~64 matchups at ~35 min each).

In [4]:
# Checks both agents move, the wall triggers are split, and Unity is not throwing on
# step_penalty. A frozen opponent still writes a full-length JSON and exits cleanly, which
# is how two 64-matchup batches were lost.
health = tc.smoke_test(
    model_a_run_dir=MODELS_ROOT / RUN_IDS[0],
    model_b_run_dir=MODELS_ROOT / RUN_IDS[1],
    octagon_dir=OCTAGON_DIR,
    unity_env_path=UNITY_ENV_PATH,
    out_root=OUT_ROOT,
    trial_seq=TRIAL_SEQ,
    episodes=8,
    base_port=tc.SMOKE_TEST_BASE_PORT,   # 11000, clear of training (5005+20*i, 16 envs
                                         # each) and of the round-robin (9100+)
)

['mlagents-learn', '/home/tom/Unity/Octagon/simulations/tournaments/260723/tournament_config.yaml', '--run-id', 'tournament/260723_0034__vs__260723_0039', '--resume', '--inference', '--base-port', '11000', '--env', '/home/tom/Unity/Octagon/builds/260909/build_260909_01_tournament_nosteppenalty.x86_64', '--no-graphics', '--env-args', '--sim_out', '/home/tom/Unity/Octagon/simulations/tournaments/260723/_smoke_test', '--sim_eps', '8', '--trial_seq', '/home/tom/repos/agent_training/trial_sequences/trials_100000_seed17.json', '--seed', '17']
smoke-test log: /home/tom/Unity/Octagon/simulations/tournaments/260723/_smoke_test/mlagents_stdout.log

trials:           8
unique positions: P1 3217, P2 3166
wall triggers:    P1 4, P2 4
step_penalty exceptions in Unity log: 0

SMOKE TEST PASSED — safe to run the full round-robin.


## Full round-robin

Every unordered pair, **including self-matchups**, and (with `INCLUDE_MIRRORS`) each pair in both seat orders. One JSON per matchup under `OUT_ROOT/<A>__vs__<B>/`. Failures are caught per-matchup (that entry is `None`) so one bad run doesn't kill the batch.

**Self-matchups vs mirrors:**
- **Self-matchups** (a model against a copy of itself) are run, one per entrant, as a ~50% baseline check.
- **Mirrors** (the same pairing with the P1/P2 seats swapped) are off in the function by default. Which seat a model takes should be irrelevant: agents aren't repositioned at trial start, so each agent's position is emergent from its own behaviour, and the only P1/P2 difference is administrative (arena setup and logging). They are on here to double the data per pair and to re-confirm that the seat has no effect.

In [ ]:
logs = run_tournament(
    models_root=MODELS_ROOT,
    octagon_dir=OCTAGON_DIR,
    unity_env_path=UNITY_ENV_PATH,
    out_root=OUT_ROOT,
    episodes=EPISODES,
    run_ids=RUN_IDS,            # None => all models under MODELS_ROOT
    include_self_matchups=INCLUDE_SELF_MATCHUPS,
    include_mirrors=INCLUDE_MIRRORS,
    trial_seq=TRIAL_SEQ,
    base_port=9100,             # ports 9100 + 20*i -> 9100..10360. Training uses 5005+20*i
                                # per model, so 9100 clears sessions up to ~206 models. Also
                                # above tensorboard (6006) and mDNS (5353).
)

In [ ]:
# Which matchups ran correctly? A matchup can exit cleanly with a full-length JSON and still
# be void: before the 260729 fix a missing environment_parameters block froze P2 in all 64
# matchups. Only suspect matchups are printed.
suspect = tc.check_batch(
    out_root=OUT_ROOT,
    tournament_ids=logs.keys(),
    expected_trials=EPISODES,
    octagon_dir=OCTAGON_DIR,
    logs=logs,                              # so matchups that raised are reported too
)

## Rerun specific matchups

For matchups that failed or came out short above, without re-running the whole batch. List the ordered `(P1, P2)` pairs in `RERUN` — the same ids that make up the matchup dir name `<P1>__vs__<P2>`.

Everything else (build, checkpoints, trial sequence, sampled actions) is identical to the round-robin, so the reruns are drop-in replacements for the originals.

**On stalled matchups.** In `260715` the `0002` self-matchup stalled rather than crashed: `No episode was completed since last summary` in its stdout, and 68 `trial start` events in ~7200 s against ~350 in ~2200 s for a healthy matchup. Both copies wandered without triggering a wall. Actions are sampled, so changing `seed` re-rolls the trajectories and is the lever most likely to change the outcome. Raising `timeout_s` alone tends to buy a longer stall. Check the trial count in the health line, not just the absence of an exception.

In [ ]:
# Ordered (P1, P2) run_id pairs to rerun. Same id twice => self-matchup.
RERUN = [
    # ("260723_0034", "260723_0034"),
]

rerun_logs = tc.rerun_matchups(
    pairs=RERUN,
    models_root=MODELS_ROOT,
    octagon_dir=OCTAGON_DIR,
    unity_env_path=UNITY_ENV_PATH,
    out_root=OUT_ROOT,          # same dir as the round-robin, so reruns land beside siblings
    episodes=EPISODES,          # trials ~= episodes - 1
    trial_seq=TRIAL_SEQ,
    seed=17,                    # change to re-roll the sampled trajectories
    timeout_s=5000,             # wall-clock cap per matchup (healthy 350-ep run: ~2200 s)
    base_port=10500,            # clear of the round-robin's 9100..10360
    archive_existing=True,      # move the old matchup dir to OUT_ROOT/../superseded/ first
)

## Resume an interrupted batch

`run_tournament` always starts at matchup 1, so re-running it after an interruption redoes
work that is already done. Worse, `run_eval` deletes `DONE.txt` before launching, so a
restart that dies early strips the marker off a matchup that had finished. That happened on
2026-09-10: the batch was interrupted at matchup 55, restarted, and overwrote the first
matchup's `DONE.txt` before dying. The 350-trial JSON survived and the fragment is in
`tournaments/superseded/`.

Use this cell instead. `outstanding_matchups` reads what is on disk and returns only the
pairs still needed, so it is safe to run repeatedly until it reports nothing outstanding.

In [11]:
# Pairs still needed: missing dir, no JSON, no DONE.txt, or failing the health check.
todo = tc.outstanding_matchups(
    out_root=OUT_ROOT,
    run_ids=RUN_IDS,
    expected_trials=EPISODES,
    include_mirrors=INCLUDE_MIRRORS,
    include_self_matchups=INCLUDE_SELF_MATCHUPS,
)

rerun_logs = tc.rerun_matchups(
    pairs=todo,
    models_root=MODELS_ROOT,        # still the frozen 3e6 snapshot
    octagon_dir=OCTAGON_DIR,
    unity_env_path=UNITY_ENV_PATH,
    out_root=OUT_ROOT,
    episodes=EPISODES,
    trial_seq=TRIAL_SEQ,
    seed=17,
    timeout_s=5000,
    base_port=10500,                # clear of the round-robin's 9100..10360
    archive_existing=True,          # partial dirs move to ../superseded/ before the rerun
)

49/56 matchups complete, 7 outstanding
  260723_0023__vs__260723_0001  (no dir)
  260723_0033__vs__260723_0051  (no dir)
  260723_0051__vs__260723_0033  (no dir)
  260723_0033__vs__260723_0023  (no dir)
  260723_0023__vs__260723_0033  (no dir)
  260723_0051__vs__260723_0023  (no dir)
  260723_0023__vs__260723_0051  (no dir)

=== [1/7] Rerun: 260723_0023 (P1) vs 260723_0001 (P2) ===
['mlagents-learn', '/home/tom/Unity/Octagon/simulations/tournaments/260723/tournament_config.yaml', '--run-id', 'tournament/260723_0023__vs__260723_0001', '--resume', '--inference', '--base-port', '10500', '--env', '/home/tom/Unity/Octagon/builds/260909/build_260909_01_tournament_nosteppenalty.x86_64', '--no-graphics', '--env-args', '--sim_out', '/home/tom/Unity/Octagon/simulations/tournaments/260723/260723_0023__vs__260723_0001', '--sim_eps', '350', '--trial_seq', '/home/tom/repos/agent_training/trial_sequences/trials_100000_seed17.json', '--seed', '17']
=== Finished: 260723_0023__vs__260723_0001 -> /home/t

In [ ]:
# Confirm the batch is complete. Expect "0 outstanding" and every matchup valid.
tc.outstanding_matchups(
    out_root=OUT_ROOT,
    run_ids=RUN_IDS,
    expected_trials=EPISODES,
    include_mirrors=INCLUDE_MIRRORS,
    include_self_matchups=INCLUDE_SELF_MATCHUPS,
)

ids = ["__vs__".join(p) for p in
       tc.expected_matchups(RUN_IDS, INCLUDE_MIRRORS, INCLUDE_SELF_MATCHUPS)]
suspect = tc.check_batch(OUT_ROOT, ids, expected_trials=EPISODES, octagon_dir=OCTAGON_DIR)